# DICE Model Economic & Emissions Part
**Exercise Session Resource Economics (Spring Term 2026)** \
Raul Hochuli (raul.hochuli@unibas.ch)

This is the first part of teaching the build up of the DICE model. **Please try to complete the tasks below before class on Monday**. I ask you once again to email my any solutions you have until Sunday evening before the class, so I can try to adjust my teaching accordingly. \
The following code has been replicated and refactored for this course based on the inital material from [Hazem Krichene](https://github.com/hazem2410) in his public Github repository [PYDICE](https://github.com/hazem2410/PyDICE), which builds on the [DICE (Dynamic Integrated Climate Economy) Model of William D. Nordhaus](https://williamnordhaus.com/dicerice-models). 

In [ ]:
# Cell should only apply when running this notebook on google COLAB
# > asks for access to Drive and permission to read and write files
path_where_notebook_is_stored_on_GDrive = 'MyDrive/1_UNIBAS_phd/ResEcon25/DICE'
import os as os

# if 'content' in os.getcwd():
#   from google.colab import drive
#   drive.mount('/content/drive')
#   os.chdir(f'/content/drive/{path_where_notebook_is_stored_on_GDrive}')


In [ ]:
# Packages used in the notebook
import os as os
import glob
import numpy as np
import pandas as pd
import glob
import datetime as dt

import plotly.graph_objects as go

from plotly.subplots import make_subplots

## Preamble

Before diving into an actual optimization of the DICE model, we first build the **Economic Part** and the **Emissions Part** of the model seperately. The goal is to construct (almost) all relationships that describe how the main state variables interact. Instead of an actuall optimization, we simply observe how given **exogenous variables** translate / propagate into **endogenous state variables** through the model’s equations. Exogenous variables are based on assumptions and are not influenced by the model itself; for now, we assume they remain constant over time. The resulting visualizations are not yet economically meaningful, but they help verify that the model dynamics run correctly. Because the model is still incomplete and split into separate components, we also do **not yet implement it as a single function**.


## 1 DICE Model - Economic Part

You should try to complete the tasks indicated by `<...>` comments in the code and gradually refine / extend the code.

**Note**: 
- Eventually, you should receive a results data frame, which contains `Prod_gross` a gross production array, which is an important model component for later usage. 
- Think of this model part as telling you: "How much global gross production is possible for the given assumed exogenous state variables?"

1. Use the social discount rate `social_time_pref_rate` to make an social discount array `social_time_pref_array` $\frac{1}{(1+\rho)^t}$ to be later applied in the calculations. Watch out that $t$ is denoted in 5 years steps. 
2. Define the exogenous variables for total factor productivity `TFPr`and decarbonization rate `Dcrb` as constants of the initial value (`TFPr0`, `Dcrb0`)
3. Complete the discounted utility `fUtil_cons_disc()` (welfare function $W_t$ in the lecture, p23/28). 
4. Add a capital depreciation rate `depr_Capt` of 0.1 to the function `fCapt()`defining the capital stock
5. Add a constant savings rate of $S_t = 0.08$ for all $t$ to the model such that investments are possible and the entire production is not just consumed. 

In [ ]:
# PARAMETERS --------------------------------------------------------------------------------
T_end   = 100
tstep   = 5
t = np.arange(1, T_end + 1)
NT = len(t)


# elasticities
elast_mrg_cons_utility = 1.45
elast_capt_in_prod     = 0.300

# discounting
social_time_pref_rate  = 0.015
# social_time_pref_array =  # <.1.>



# EXOGENOUS STATE VARIABLES -------------------------------------------------------------------------------------------
# labour (Labr) 
Labr0 = 7403    # inital world population (millions) 2015
Labr = np.full(NT, Labr0)

# total factor productivity (TFP)
TFPr0 = 5.115   # initial level of total factor productivity 2015
# TFPr = # <.2.>

# decarbonization rate (Dcrb)
Dcrb0 = 0.35032
# Dcrb =    # <.2.>

# carbon land emissions (CEms_land)
CEms_land0 = 2.6   # initial carbon emissions from land use 2015, (2.6 GtCO2/yr)
CEms_land = np.full(NT, CEms_land0)   

# abatement cost (cost1_Abat)
Bstp0 = 550  # Cost of backstop technology 2010$ per ton CO2
cost1_Abat = np.full(NT, Bstp0)  


# * extension during class *
# instead of constants, you can also import a CSV file containing all the exogenous variables
# exog_Vars_DICE = pd.read_csv(f'input/DICE_exogn_vars_default.csv')# read exogenous variables from input file
# TFPr       = exog_Vars_DICE['TFPr'].values
# Labr       = exog_Vars_DICE['Labr'].values
# Dcrb       = exog_Vars_DICE['Dcrb'].values
# gr_Dcrb    = exog_Vars_DICE['gr_Dcrb'].values
# cost1_Abat = exog_Vars_DICE['cost1_Abat'].values  
# CEms_land  = exog_Vars_DICE['CEms_land'].values




# ENDOGENOUS STATE VARIABLES -------------------------------------------------------------------------------------------

# production
Prod_gross = np.zeros(NT)
def fProd_gross(iTFPr, iLabr, iCapt, idx):
    return iTFPr[idx] * ((iLabr[idx]/1000) ** (1 - elast_capt_in_prod)) * (iCapt[idx] ** elast_capt_in_prod)

# investment 
Invs = np.zeros(NT)
def fInvs(iSavi, iProd_gross, idx):
    return iSavi[idx] * iProd_gross[idx] # <.5.>

# capital
Capt0           = 223.3
# depr_Capt       = 0.1     # <.4.>
Capt            = np.zeros(NT)
Capt[0]         = Capt0
def fCapt(iCapt, iInvs, idx):
    if (idx == 0):
        return Capt0
    else:
        # return ... + tstep * iInvs[idx-1]   # <.4.>

# consumption 
Cons = np.zeros(NT)
def fCons(iProd_gross, iInvs, idx):
    return iProd_gross[idx] - iInvs[idx]

Cons_pcap = np.zeros(NT)
def fCons_pcap(iCons, iLabr, idx):
    return 1000 * iCons[idx] / iLabr[idx]

# utility
Util_pcap_cons = np.zeros(NT)
def fUtil_pcap_cons(iCons, iLabr, idx):
    return ((iCons[idx] * 1000 / iLabr[idx]) ** (1 - elast_mrg_cons_utility) -1) / (1 - elast_mrg_cons_utility) - 1  

Util_cons_disc = np.zeros(NT)
def fUtil_cons_disc(iUtil_pcap_cons, iLabr, idx):
    # return social_time_pref_array    # <.3.> 



# DECISION VARIABLS ------------------------------------------------------------ 
# ("economic" decison variable is the savings rate)
# Savi = ... # <.5.> 



# OBJECTIVE FUNCTION ------------------------------------------------------------ 
# (final utility function as objective funcstion still missing)
for i in range(NT):
    Capt[i] = fCapt(Capt, Invs, i)
    Prod_gross[i] = fProd_gross(TFPr, Labr, Capt, i)
    Invs[i] = fInvs(Savi, Prod_gross, i)
    Cons[i] = fCons(Prod_gross, Invs, i)
    Cons_pcap[i] = fCons_pcap(Cons, Labr, i)
    Util_pcap_cons[i] = fUtil_pcap_cons(Cons, Labr, i)
    Util_cons_disc[i] = fUtil_cons_disc(Util_pcap_cons, Labr, i)



# OPTIMIZATION ------------------------------------------------------------ 
# (still missing because we only look first how exogenous values propagate through model mechanics)



# EXPORT RESULTS ------------------------------------------------------------
results_df = pd.DataFrame({
    't':                t, 
    't_year':           2000 + t*tstep,
    'Savi':             Savi,
    'TFPr':             TFPr,
    'Labr':             Labr,
    'Dcrb':             Dcrb,
    'cost1_Abat':       cost1_Abat,
    'CEms_land':        CEms_land,
    'Capt':             Capt,
    'Prod_gross':       Prod_gross,
    'Invs':             Invs,
    'Cons':             Cons,
    'Cons_pcap':        Cons_pcap,
    'Util_pcap_cons':   Util_pcap_cons,
    'Util_cons_disc':   Util_cons_disc,
})


# VISUALIZATION ------------------------------------------------------------
cols_to_plot = [col for col in results_df.columns if col not in ['t', 't_year']]
n_rows = len(cols_to_plot) // 3 + 1 if len(cols_to_plot) % 3 > 0 else len(cols_to_plot) // 3
fig = make_subplots(rows=n_rows, cols =3, subplot_titles=cols_to_plot, vertical_spacing=0.12)
for i, col in enumerate(cols_to_plot):
    row_idx = i // 3 + 1
    col_idx = i % 3 + 1
    if col in ['Savi', 'ECtr']:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, mode='lines+markers'), row=row_idx, col=col_idx)
        # fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, line = dict(dash = 'dash', width = 3)), row=row_idx, col=col_idx)
    elif col in ['TFPr', 'Labr', 'Dcrb', 'gr_Dcrb', 'cost1_Abat', 'CEms_land', ]:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, line = dict(dash = 'dot')), row=row_idx, col=col_idx)
    else:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col), row=row_idx, col=col_idx)
    fig.update_xaxes(title_text="Year", row=row_idx, col=col_idx)

fig.update_layout(title_text="Economic State variables", template = 'plotly_white', width = 2400, height = 800)
fig.show()


## 2 DICE Model - Emissions Part

After having completed the economic part of the DICE Model, let's consider the emission part. The emission part has more complex functional terms defining its state variables but follows exaclty the same structure as the economic part. Try to complete the tasks below (indicated as `<..>` comments in the code) and refine the emission part of the model step by step. 

Note:
- `Prod_gross` is here the state variables direclty influencing the amount of emissions `CEms` which trickles down through the emission part of the model. If you were able to solve the economic part, use the values for `Prod_gross` you got there, if not just use a flat constant given below. 
- Think of this model part as telling you: "What is the emission level given a certain global gross production level, and how do these emissions cause econoimc damages by propagating through the different carbon stocks and temperature changes?"

Tasks: 
1. Build the function `fDamg_frac()` defining the fractional damages of the atmosphere temperature `iTemp_atmo` ($a_1 \cdot T_{AT,t} + a_2 \cdot T_{AT,t}^{a_3}$)
2. Complete the missing elements of the carbon cycle transition matrix (hint 1: $b_{11} = 1-b_{12}$; hint 2: $b_{32} = b_{23}(M_{LO}^{*}/M_{UP}^{*})$)
3. Complete the function `fCStk_atmo()` defining the carbon stock in the atmosphere
4. Complete the function `fCStk_ocup()` defining the carbon stock in the upper ocean layer
5. Add an constant emission control rate `ECtr` of 0.2 to the model and rerun the state variable definitions. 
6. Shift the functional assignment of the `RFor` state variable up in the order of the "objective function" so that it get's assigned after the carbon emissions for industry and overall emissions (`CEms`). Does the "model" still work? 

In [ ]:
# PARAMETERS --------------------------------------------------------------------------------
T_end   = 100
tstep   = 5
t = np.arange(1, T_end + 1)
NT = len(t)


# GET PRODUCTION VARS -------------------------------------------------------------------------------------------
Prod_gross = np.full(NT, 108)
# OR 
Prod_gross = results_df['Prod_gross'] 



# EXOGENOUS VARS -------------------------------------------------------------------------------------------
# labour (Labr) 
Labr0 = 7403    # inital world population (millions) 2015
Labr = np.full(NT, Labr0)

# total factor productivity (TFP)
TFPr0 = 5.115   # initial level of total factor productivity 2015
TFPr = np.full(NT, TFPr0)  

# decarbonization rate (Dcrb)
Dcrb0 = 0.35032
Dcrb = np.full(NT, Dcrb0)  

# carbon land emissions (CEms_land)
CEms_land0 = 2.6   # initial carbon emissions from land use 2015, (2.6 GtCO2/yr)
CEms_land = np.full(NT, CEms_land0)   

# abatement cost (cost1_Abat)
Bstp0 = 550  # Cost of backstop technology 2010$ per ton CO2
cost1_Abat = np.full(NT, Bstp0)  

# * extansion *
# instead of constants, you can also import a CSV file containing all the exogenous variables
# exog_Vars_DICE = pd.read_csv(f'input/DICE_exogn_vars_default.csv')# read exogenous variables from input file
# TFPr       = exog_Vars_DICE['TFPr'].values
# Labr       = exog_Vars_DICE['Labr'].values
# Dcrb       = exog_Vars_DICE['Dcrb'].values
# gr_Dcrb    = exog_Vars_DICE['gr_Dcrb'].values
# cost1_Abat = exog_Vars_DICE['cost1_Abat'].values  
# CEms_land  = exog_Vars_DICE['CEms_land'].values



# ENDOGENOUS VARS -------------------------------------------------------------------------------------------

# carbon emissions (industry and total)
CEms_indu = np.zeros(NT)
def fCEms_indu(iProd_gross, iECtr, iDcrb, idx):
    return iDcrb[idx] * iProd_gross[idx] * (1 - iECtr[idx])

CEms = np.zeros(NT)
def fCEms(iCEms_indu, iCEms_land, idx):
    return iCEms_indu[idx] + iCEms_land[idx]

# carbon cycle ocean transition matrix
CStk_atmo0    =  851                      # Initial Concentration in atmosphere 2015 (GtC)
CStk_ocup0    =  460                      # Initial Concentration in upper strata 2015 (GtC)
CStk_oclo0    =  1740                     # Initial Concentration in lower strata 2015 (GtC)
CStk_atmo_eq  =  588                      # Equilibrium concentration atmosphere (GtC)
CStk_ocup_eq  =  360                      # Equilibrium concentration upper strata (GtC)
CStk_oclo_eq  =  1720                     # Equilibrium concentration lower strata (GtC)
b12           =  0.12                     # carbon cycle transition matrix (atmo to up)
b23           =  0.007                    # carbon cycle transition matrix (up to lo)

# b11 = ...  # <.2.>
b21 = b12*(CStk_ocup_eq/CStk_atmo_eq)   
b22 = 1 - b21 - b23
# b32 = ... # <.2.>
b33 = 1 - b32

CStk_atmo       = np.zeros(NT)
CStk_atmo[0]    = CStk_atmo0
def fCStk_atmo(iCStk_atmo, iECtr, iCEms, idx):
    if (idx == 0):
        return CStk_atmo0
    else:
        # return ... + iCEms[idx-1]*5/3.666    # <.3.>

CStk_ocup       = np.zeros(NT)
CStk_ocup[0]    = CStk_ocup0
def fCStk_ocup(iCStk_atmo, iCStk_ocup, iCStk_oclo, idx):
    if (idx == 0):
        return CStk_ocup0
    else:
        # return ...   # <.4.>

CStk_oclo       = np.zeros(NT)
CStk_oclo[0]    = CStk_oclo0
def fCStk_oclo(iCStk_ocup, iCStk_oclo, idx):
    if (idx == 0):
        return CStk_oclo0
    else:
        return iCStk_oclo[idx-1]*b33 + iCStk_ocup[idx-1]*b23
    
# radiative forcing (RFor)
incr_RFor_dbl_crbn = 3.6813                   # Forcing from doubling CO2 (Wm-2) 2015
incr_temp_dbl_crb = 3.1                      # Equilibrium temp impact (oC per doubling CO2)
CStk_atmo1750 =  588                      # Initial Concentration in 1750, 

RFor = np.zeros(NT)
def fRFor(iCStk_atmo, idx):
    return incr_RFor_dbl_crbn * np.log(iCStk_atmo[idx] / CStk_atmo1750) / np.log(2) 

# temperature
Temp_atmo0 = 0.85                    # Initial atmospheric temp change (C from 1900)
Temp_ocea0 = 0.0068                   # Initial lower stratum temp change (C from 1900)
c1         = 0.1005                   # Climate equation coefficient for upper level
c3         = 0.088                    # Transfer coefficient upper to lower stratum
c4         = 0.025                    # Transfer coefficient for lower level

Temp_atmo       = np.zeros(NT)
Temp_atmo[0]    = Temp_atmo0
def fTemp_atmo(iTemp_atmo, iRFor, iTemp_ocea, idx):
    if (idx == 0):
        return Temp_atmo0
    else:
        return iTemp_atmo[idx-1] + c1*(iRFor[idx] - (incr_RFor_dbl_crbn/incr_temp_dbl_crb)*iTemp_atmo[idx-1] - c3*(iTemp_atmo[idx-1] - iTemp_ocea[idx-1])) 

Temp_ocea       = np.zeros(NT)
Temp_ocea[0]    = Temp_ocea0
def fTemp_ocea(iTemp_atmo, iTemp_ocea, idx):
    if (idx == 0):
        return Temp_ocea0
    else:
        return iTemp_ocea[idx-1] + c4*(iTemp_atmo[idx-1] - iTemp_ocea[idx-1])

# damages
a1 = 0                       # Damage function intercept
a2 = 0.00236               # Damage function quadratic term
a3 = 2.00                  # Damage function exponent

Damg_frac = np.zeros(NT)
def fDamg_frac(iTemp_atmo, idx):
    # return ...  # <.1.>

Damg = np.zeros(NT)
def fDamg(iProd_gross, iDamg_frac, idx):
    return iProd_gross[idx] * iDamg_frac[idx]


# abatement cost
cost2_Abat = 2.6                    

Abat_cost = np.zeros(NT)
def fAbat_cost(iProd_gross, iECtr, icost1_Abat, idx):
    return iProd_gross[idx] * icost1_Abat[idx] * iECtr[idx] ** cost2_Abat

Abat_MC = np.zeros(NT)
Bstp = np.full(NT, Bstp0)  
def fAbat_MC(iECtr, idx):
    return Bstp[idx] * iECtr[idx] ** (cost2_Abat-1)



# DECISION VARIABLS ------------------------------------------------------------ 
# ("emission" decison variable is the emission control rate)
# ECtr = np.full(NT, ...) # # <.5.>



# OBJECTIVE FUNCTION ------------------------------------------------------------ 
# (not true strictly speaking because we do not optimize over any of these components, just for similar model strcutrue to emissions part)
for i in range(NT): 
    CEms_indu[i] = fCEms_indu(Prod_gross, ECtr, Dcrb, i)
    CEms[i]      = fCEms(CEms_indu, CEms_land, i)
    # <.6.> to here
    Abat_cost[i] = fAbat_cost(Prod_gross, ECtr, cost1_Abat, i)
    Abat_MC[i]   = fAbat_MC(ECtr, i)
    CStk_atmo[i] = fCStk_atmo(CStk_atmo, ECtr, CEms, i)
    CStk_ocup[i] = fCStk_ocup(CStk_atmo, CStk_ocup, CStk_oclo, i)
    CStk_oclo[i] = fCStk_oclo(CStk_ocup, CStk_oclo, i)
    # <.6.> from here
    RFor[i]      = fRFor(CStk_atmo, i)
    Temp_atmo[i] = fTemp_atmo(Temp_atmo, RFor, Temp_ocea, i)
    Temp_ocea[i] = fTemp_ocea(Temp_atmo, Temp_ocea, i)
    Damg_frac[i] = fDamg_frac(RFor, i)
    Damg[i]      = fDamg(Prod_gross, Damg_frac, i)


# OPTIMIZATION ------------------------------------------------------------ 
# (still missing because we only look first how exogenous values propagate through model mechanics)



# EXPORT RESULTS ------------------------------------------------------------
results_df = pd.DataFrame({
    't':                t, 
    't_year':           2000 + t*tstep,
    'ECtr':            ECtr,
    'TFPr':             TFPr,
    'Labr':             Labr,
    'Dcrb':             Dcrb,
    'cost1_Abat':       cost1_Abat,
    'CEms_land':        CEms_land,
    'Prod_gross':       Prod_gross,
    'CEms':             CEms,
    'CEms_indu':        CEms_indu,
    'RFor':             RFor,
    'Abat_cost':        Abat_cost,
    'Abat_MC':          Abat_MC,
    'CStk_atmo':       CStk_atmo,
    'CStk_ocup':       CStk_ocup,
    'CStk_oclo':       CStk_oclo,
    'Temp_atmo':       Temp_atmo,
    'Temp_ocea':       Temp_ocea,
    'Damg_frac':       Damg_frac,
    'Damg':            Damg,

})



cols_to_plot = [col for col in results_df.columns if col not in ['t', 't_year']]
n_rows = len(cols_to_plot) // 3 + 1 if len(cols_to_plot) % 3 > 0 else len(cols_to_plot) // 3
fig = make_subplots(rows=n_rows, cols =3, subplot_titles=cols_to_plot, vertical_spacing=0.12)
for i, col in enumerate(cols_to_plot):
    row_idx = i // 3 + 1
    col_idx = i % 3 + 1
    if col in ['Savi', 'ECtr']:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, mode='lines+markers'), row=row_idx, col=col_idx)
        # fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, line = dict(dash = 'dash', width = 3)), row=row_idx, col=col_idx)
    elif col in ['TFPr', 'Labr', 'Dcrb', 'gr_Dcrb', 'cost1_Abat', 'CEms_land', ]:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, line = dict(dash = 'dot')), row=row_idx, col=col_idx)
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col, line = dict(dash = 'dot')), row=row_idx, col=col_idx)
    else:
        fig.add_trace(go.Scatter(x=results_df['t_year'], y=results_df[col], name=col), row=row_idx, col=col_idx)
    fig.update_xaxes(title_text="Year", row=row_idx, col=col_idx)


fig.update_layout(title_text="Emission State variables", template = 'plotly_white', width = 2400, height = 800)
fig.show()
